# Cross-Institutional Transfer: Figures & Tables

Produces publication-quality visuals for Section 2.3.

- **Figure 5**: Local vs Transfer dumbbell plot (main text)
- **Table S**: Full transfer results (supplementary)
- **Table S**: Per-class transfer breakdown (supplementary)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

# ── Figure rcParams (matches fig_selection_efficiency) ──
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'legend.fontsize': 7.5,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'lines.linewidth': 1.3,
    'lines.markersize': 4,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})


In [ ]:
REPO_ROOT = Path('.').resolve()
root_candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next(
    (c.resolve() for c in root_candidates
     if (c / 'results').exists() and (c / 'notebooks').exists()),
    Path.cwd().resolve(),
)
print(f'Using repo root: {REPO_ROOT}')

OUTPUT_DIR = REPO_ROOT / 'notebooks'
OUTPUT_DIR.mkdir(exist_ok=True)

# Load consolidated tables (from main experiments)
CONSOLIDATED_DIR = REPO_ROOT / 'analysis' / 'consolidated_loo_tables'
assert CONSOLIDATED_DIR.exists(), f'Missing {CONSOLIDATED_DIR}'
table_1_main = pd.read_csv(CONSOLIDATED_DIR / 'table_1.csv')

# Load cross-institutional tables
CROSS_DIR = REPO_ROOT / 'analysis' / 'cross_institutional_tables'
assert CROSS_DIR.exists(), f'Missing {CROSS_DIR}'
transfer_table_1 = pd.read_csv(CROSS_DIR / 'transfer_table_1.csv')
transfer_table_2 = pd.read_csv(CROSS_DIR / 'transfer_table_2.csv')
transfer_table_3 = pd.read_csv(CROSS_DIR / 'transfer_table_3.csv')
transfer_table_4 = pd.read_csv(CROSS_DIR / 'transfer_table_4.csv')

print(f'table_1_main:      {len(table_1_main)} rows')
print(f'transfer_table_1:  {len(transfer_table_1)} rows')
print(f'transfer_table_2:  {len(transfer_table_2)} rows')
print(f'transfer_table_3:  {len(transfer_table_3)} rows')


In [ ]:
MODEL_DISPLAY_NAMES = {
    'meta-llama/Llama-3.2-3B-Instruct': 'Llama-3.2-3B',
    'mistralai/Ministral-3-3B-Instruct-2512-BF16': 'Ministral-3B',
    'google/gemma-3-4b-it': 'Gemma-3-4B',
    'Qwen/Qwen3-4B-Instruct-2507': 'Qwen3-4B',
    'microsoft/MediPhi-Instruct': 'MediPhi',
    'microsoft/Phi-3.5-mini-instruct': 'Phi-3.5-mini',
    'microsoft/Phi-4-mini-instruct': 'Phi-4-mini',
    'COMMITTEE': 'Committee',
    'regex_baseline': 'Regex baseline',
}

# Same order as other figures: weakest zero-shot → strongest, then committee
MODEL_ORDER = [
    'meta-llama/Llama-3.2-3B-Instruct',
    'microsoft/Phi-4-mini-instruct',
    'microsoft/MediPhi-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
    'mistralai/Ministral-3-3B-Instruct-2512-BF16',
    'google/gemma-3-4b-it',
    'Qwen/Qwen3-4B-Instruct-2507',
    'COMMITTEE',
]

COLORS = {
    'zero_shot':  '#111111',
    'label_only': '#0072B2',
    'rationale':  '#009E73',
    'random':     '#6c757d',
    'regex':      '#D55E00',
    'WithUpdate': '#D55E00',
    'Random':     '#6c757d',
    'sig_pos':    '#009E73',   # significant positive (local better)
    'sig_neg':    '#D55E00',   # significant negative (transfer better)
    'nonsig':     '#999999',
    'local':      '#009E73',
    'transfer':   '#0072B2',
}

def short_name(model):
    return MODEL_DISPLAY_NAMES.get(model, model)

def get_f1_transfer(df, cohort, prompt_variant, model, regime, prompt_regime):
    mask = ((df['cohort'] == cohort) & (df['model'] == model) &
            (df['regime'] == regime) & (df['prompt_regime'] == prompt_regime) &
            (df['prompt_variant'] == prompt_variant))
    rows = df[mask]
    return rows.iloc[0]['macro_f1'] if len(rows) > 0 else np.nan

def get_sig_transfer(df, cohort, prompt_variant, model, comparison):
    mask = ((df['cohort'] == cohort) & (df['prompt_variant'] == prompt_variant) &
            (df['model'] == model) & (df['comparison'] == comparison))
    rows = df[mask]
    if len(rows) == 0:
        return False, 0.0
    return bool(rows.iloc[0]['sig']), float(rows.iloc[0]['delta_f1'])


---
## Figure 5: Cross-Institutional Transfer (Main Text)

Dumbbell plot: local exemplars (green circle) vs transfer exemplars (blue triangle) per model.

Rationale-augmented ICL, long prompt, m=4.

Two panels: (a) MIMIC-IV (Indian → MIMIC), (b) Indian OCR (MIMIC → Indian).

Green lines = local significantly better; orange lines = transfer significantly better; grey = non-significant.

In [ ]:
PROMPT_VARIANT = 'long'
PROMPT_REGIME = 'rationale-augmented ICL'
COMPARISON_KEY = 'local_rationale vs transfer_rationale'

fig, axes = plt.subplots(
    1, 2,
    figsize=(7.2, 3.2),
    sharey=True,
    gridspec_kw={'wspace': 0.08},
)

for ax, panel_label, cohort, title, transfer_label in [
    (axes[0], 'a', 'MIMIC', 'MIMIC-IV ($n = 628$)', 'Indian \u2192 MIMIC'),
    (axes[1], 'b', 'Indian', 'Indian OCR ($n = 340$)', 'MIMIC \u2192 Indian'),
]:
    y_positions = np.arange(len(MODEL_ORDER))
    all_points = []

    for i, model in enumerate(MODEL_ORDER):
        # Local F1 (WithUpdate, rationale)
        local_f1 = get_f1_transfer(transfer_table_1, cohort, PROMPT_VARIANT,
                                    model, 'WithUpdate', PROMPT_REGIME)
        # Transfer F1
        trans_f1 = get_f1_transfer(transfer_table_1, cohort, PROMPT_VARIANT,
                                    model, 'Transfer', PROMPT_REGIME)

        # Significance and direction
        sig, delta = get_sig_transfer(transfer_table_3, cohort, PROMPT_VARIANT,
                                       model, COMPARISON_KEY)

        # Alternating background bands
        if i % 2 == 0:
            ax.axhspan(i - 0.4, i + 0.4, color='#f7f9fb', zorder=0)

        # Connecting line + delta annotation only for significant differences
        if not np.isnan(local_f1) and not np.isnan(trans_f1):
            if sig:
                line_color = COLORS['sig_pos'] if delta > 0 else COLORS['sig_neg']
                ax.plot(
                    [local_f1, trans_f1], [i, i],
                    color=line_color, linewidth=2.0,
                    solid_capstyle='round', zorder=2,
                )
                # Delta annotation
                sign = '+' if delta > 0 else ''
                mid_x = max(local_f1, trans_f1) + 0.01
                ax.text(
                    mid_x, i,
                    f'{sign}{delta:.2f}',
                    fontsize=5, color=line_color,
                    va='center', ha='left', fontweight='bold',
                    clip_on=False,
                )

        # Local dot (green circle)
        if not np.isnan(local_f1):
            ax.scatter(
                local_f1, i, color=COLORS['local'],
                marker='o', s=28, zorder=4,
                edgecolors='white', linewidths=0.4,
            )
            all_points.append(local_f1)

        # Transfer dot (blue triangle)
        if not np.isnan(trans_f1):
            ax.scatter(
                trans_f1, i, color=COLORS['transfer'],
                marker='^', s=28, zorder=3,
                edgecolors='white', linewidths=0.4,
            )
            all_points.append(trans_f1)

    # Regex baseline
    regex_rows = table_1_main[
        (table_1_main['cohort'] == cohort) &
        (table_1_main['model'] == 'regex_baseline')
    ]
    if len(regex_rows) > 0:
        regex_f1 = regex_rows.iloc[0]['macro_f1']
        if not np.isnan(regex_f1):
            ax.axvline(regex_f1, color=COLORS['regex'], linestyle='--',
                       linewidth=1.0, alpha=0.6, zorder=1)
            ax.text(regex_f1, -0.6, 'regex', fontsize=5.5,
                    color=COLORS['regex'], ha='center', va='top')
            all_points.append(regex_f1)

    ax.set_yticks(y_positions)
    ax.set_yticklabels([short_name(m) for m in MODEL_ORDER], fontsize=7.5)
    ax.set_xlabel('Macro F1')
    ax.set_title(title, pad=8)
    ax.grid(axis='x', linewidth=0.3, alpha=0.5)
    ax.set_ylim(-0.8, len(MODEL_ORDER) - 0.2)
    ax.invert_yaxis()

    if all_points:
        xmin = max(0, min(all_points) - 0.05)
        xmax = min(1.08, max(all_points) + 0.09)
        ax.set_xlim(xmin, xmax)

    ax.text(-0.08, 1.04, panel_label, transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='bottom', ha='left')

# Legend — no non-significant line entry
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=COLORS['local'],
           markersize=5.5, label='Local exemplars'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor=COLORS['transfer'],
           markersize=5.5, label='Transfer exemplars'),
    Line2D([0], [0], color=COLORS['sig_pos'], linewidth=2.0,
           label='Local sig. better'),
    Line2D([0], [0], color=COLORS['sig_neg'], linewidth=2.0,
           label='Transfer sig. better'),
    Line2D([0], [0], color=COLORS['regex'], linestyle='--', linewidth=1.0,
           label='Regex baseline'),
]
axes[0].legend(
    handles=legend_elements, loc='lower left',
    frameon=True, fancybox=False, edgecolor='#cccccc',
    framealpha=0.95, borderpad=0.4, fontsize=6,
)

fig.savefig(OUTPUT_DIR / 'fig_cross_transfer.pdf')
fig.savefig(OUTPUT_DIR / 'fig_cross_transfer.png')
print(f'Saved: {OUTPUT_DIR / "fig_cross_transfer.pdf"}')
plt.show()

---
## Supplementary Table: Full Transfer Classification Results

All models × both prompt variants × label-only and rationale-augmented.

Format: F1 [95% CI]. Includes both local (WithUpdate) and Transfer columns.

In [ ]:
def format_f1(row):
    if pd.isna(row['macro_f1']):
        return '—'
    return f"{row['macro_f1']:.2f} [{row['ci_low']:.2f}, {row['ci_high']:.2f}]"

def build_supp_table(prompt_variant):
    rows = []
    for model in MODEL_ORDER:
        mname = short_name(model)
        for regime, pr in [('WithUpdate', 'label-only ICL'),
                           ('Transfer', 'label-only ICL'),
                           ('WithUpdate', 'rationale-augmented ICL'),
                           ('Transfer', 'rationale-augmented ICL')]:
            row = {'Model': mname, 'Regime': regime, 'Prompt': pr.replace(' ICL', '')}
            for cohort in ['MIMIC', 'Indian']:
                sub = transfer_table_1[
                    (transfer_table_1['cohort'] == cohort) &
                    (transfer_table_1['prompt_variant'] == prompt_variant) &
                    (transfer_table_1['model'] == model) &
                    (transfer_table_1['regime'] == regime) &
                    (transfer_table_1['prompt_regime'] == pr)
                ]
                if len(sub) > 0:
                    row[cohort] = format_f1(sub.iloc[0])
                else:
                    row[cohort] = '—'
            rows.append(row)
    return pd.DataFrame(rows)

for pv in ['long', 'short']:
    print(f"\n{'='*60}")
    print(f"Supplementary Table — {pv} prompt")
    print(f"{'='*60}")
    st = build_supp_table(pv)
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
        display(st)


---
## Supplementary Table: Per-Class F1 under Transfer (long prompt, rationale-augmented)

Shows which classes degrade most under cross-institutional exemplar transfer.

In [ ]:
def build_perclass_table():
    rows = []
    for cohort in ['MIMIC', 'Indian']:
        for model in MODEL_ORDER:
            mname = short_name(model)
            for regime in ['WithUpdate', 'Transfer']:
                sub = transfer_table_2[
                    (transfer_table_2['cohort'] == cohort) &
                    (transfer_table_2['model'] == model) &
                    (transfer_table_2['regime'] == regime) &
                    (transfer_table_2['prompt_regime'] == 'rationale-augmented ICL')
                ]
                if sub.empty:
                    continue
                row = {'Cohort': cohort, 'Model': mname, 'Source': regime}
                for _, r in sub.iterrows():
                    row[r['cls'].capitalize()] = f"{r['f1']:.3f}"
                macro = sub['f1'].mean()
                row['Macro'] = f"{macro:.3f}"
                rows.append(row)
    return pd.DataFrame(rows)

perclass = build_perclass_table()
for cohort in ['MIMIC', 'Indian']:
    print(f"\n{'='*60}")
    print(f"{cohort} — per-class F1 (long prompt, rationale-augmented)")
    print(f"{'='*60}")
    sub = perclass[perclass['Cohort'] == cohort].drop(columns=['Cohort'])
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
        display(sub)


---
## LaTeX Export — Supplementary Transfer Table

In [ ]:
def supp_transfer_latex(prompt_variant='long'):
    t = transfer_table_1
    lines = []
    lines.append(r'\\begin{table}[!ht]')
    lines.append(r'\\centering')
    lines.append(r'\\small')
    lines.append(r'\\caption{Cross-institutional transfer classification performance '
                 f'(macro F1 [95\\% bootstrap CI]) at $m = 4$ exemplars per class ($k = 12$), '
                 f'{prompt_variant} prompt variant. '
                 r'Local: exemplars drawn from the same corpus; '
                 r'Transfer: exemplars drawn from the other corpus.}')
    lines.append(r'\\label{tab:transfer_results}')
    lines.append('')

    for cohort, cohort_label in [('MIMIC', 'MIMIC-IV cohort ($n = 628$)'),
                                  ('Indian', 'Indian OCR cohort ($n = 340$)')]:
        ncols = 5
        lines.append(r'\\begin{tabular}{l' + 'c' * 4 + '}')
        lines.append(r'\\toprule')
        lines.append(r' & \\multicolumn{2}{c}{Label-only} & \\multicolumn{2}{c}{Rationale-augmented} \\\\')
        lines.append(r'\\cmidrule(lr){2-3} \\cmidrule(lr){4-5}')
        lines.append(r'Model & Local & Transfer & Local & Transfer \\\\')
        lines.append(r'\\midrule')
        lines.append(f'\\multicolumn{{{ncols}}}{{l}}{{\\textit{{{cohort_label}}}}} \\\\')
        lines.append(r'\\addlinespace[2pt]')

        for model in MODEL_ORDER:
            mname = short_name(model)
            vals = []
            for regime, pr in [('WithUpdate', 'label-only ICL'),
                                ('Transfer', 'label-only ICL'),
                                ('WithUpdate', 'rationale-augmented ICL'),
                                ('Transfer', 'rationale-augmented ICL')]:
                sub = t[(t['cohort'] == cohort) & (t['prompt_variant'] == prompt_variant) &
                        (t['model'] == model) & (t['regime'] == regime) &
                        (t['prompt_regime'] == pr)]
                if len(sub) > 0 and not pd.isna(sub.iloc[0]['macro_f1']):
                    r = sub.iloc[0]
                    vals.append(f"{r['macro_f1']:.2f} [{r['ci_low']:.2f}, {r['ci_high']:.2f}]")
                else:
                    vals.append('—')
            lines.append(f"{mname} & {' & '.join(vals)} \\\\")

        lines.append(r'\\bottomrule')
        lines.append(r'\\end{tabular}')
        lines.append(r'\\vspace{6pt}')
        lines.append('')

    lines.append(r'\\end{table}')

    latex_str = '\n'.join(lines)
    # Un-double the escapes for actual LaTeX
    latex_str = latex_str.replace('\\\\', '\\\\')
    print(latex_str)
    return latex_str

print("=== Long prompt ===")
_ = supp_transfer_latex('long')


---
## Key Statistics for Prose (Section 2.3)

In [ ]:
print("KEY NUMBERS FOR SECTION 2.3 PROSE")
print("=" * 60)

# ── MIMIC (Indian → MIMIC) ──
print("\nMIMIC cohort (Indian → MIMIC), long prompt, rationale-augmented:")
mimic_ra = transfer_table_3[
    (transfer_table_3['cohort'] == 'MIMIC') &
    (transfer_table_3['prompt_variant'] == 'long') &
    (transfer_table_3['comparison'] == 'local_rationale vs transfer_rationale')
].copy()
mimic_ra['model_short'] = mimic_ra['model'].apply(short_name)

n_sig_local = ((mimic_ra['sig']) & (mimic_ra['delta_f1'] > 0)).sum()
n_sig_transfer = ((mimic_ra['sig']) & (mimic_ra['delta_f1'] < 0)).sum()
n_nonsig = (~mimic_ra['sig']).sum()
excl_llama = mimic_ra[mimic_ra['model'] != 'meta-llama/Llama-3.2-3B-Instruct']
excl_llama_comm = excl_llama[excl_llama['model'] != 'COMMITTEE']
print(f"  Total comparisons: {len(mimic_ra)} (7 models + committee)")
print(f"  Local sig. better: {n_sig_local}")
print(f"  Transfer sig. better: {n_sig_transfer}")
print(f"  Non-significant: {n_nonsig}")
print(f"  Mean |Δ| (7 models excl. committee): {excl_llama_comm['delta_f1'].abs().mean():.3f}")
print(f"  Models with non-sig Δ: {', '.join(excl_llama[~excl_llama['sig']]['model_short'].tolist())}")

# Best transfer F1 on MIMIC (excluding Llama)
mimic_trans = transfer_table_1[
    (transfer_table_1['cohort'] == 'MIMIC') &
    (transfer_table_1['prompt_variant'] == 'long') &
    (transfer_table_1['regime'] == 'Transfer') &
    (transfer_table_1['prompt_regime'] == 'rationale-augmented ICL') &
    (~transfer_table_1['model'].isin(['meta-llama/Llama-3.2-3B-Instruct', 'COMMITTEE']))
]
best = mimic_trans.loc[mimic_trans['macro_f1'].idxmax()]
print(f"  Best transfer model: {short_name(best['model'])} F1={best['macro_f1']:.3f}")

# ── Indian (MIMIC → Indian) ──
print("\nIndian cohort (MIMIC → Indian), long prompt, rationale-augmented:")
indian_ra = transfer_table_3[
    (transfer_table_3['cohort'] == 'Indian') &
    (transfer_table_3['prompt_variant'] == 'long') &
    (transfer_table_3['comparison'] == 'local_rationale vs transfer_rationale')
].copy()
indian_ra['model_short'] = indian_ra['model'].apply(short_name)

n_sig_local_i = ((indian_ra['sig']) & (indian_ra['delta_f1'] > 0)).sum()
n_sig_transfer_i = ((indian_ra['sig']) & (indian_ra['delta_f1'] < 0)).sum()
n_nonsig_i = (~indian_ra['sig']).sum()
excl_llama_i = indian_ra[indian_ra['model'] != 'meta-llama/Llama-3.2-3B-Instruct']
excl_llama_comm_i = excl_llama_i[excl_llama_i['model'] != 'COMMITTEE']
print(f"  Total comparisons: {len(indian_ra)}")
print(f"  Local sig. better: {n_sig_local_i}")
print(f"  Transfer sig. better: {n_sig_transfer_i}")
print(f"  Non-significant: {n_nonsig_i}")
print(f"  Mean |Δ| excl Llama+Committee: {excl_llama_comm_i['delta_f1'].abs().mean():.3f}")

# Transfer F1 for committee on Indian
comm_indian = transfer_table_1[
    (transfer_table_1['cohort'] == 'Indian') &
    (transfer_table_1['prompt_variant'] == 'long') &
    (transfer_table_1['regime'] == 'Transfer') &
    (transfer_table_1['prompt_regime'] == 'rationale-augmented ICL') &
    (transfer_table_1['model'] == 'COMMITTEE')
]
if len(comm_indian) > 0:
    r = comm_indian.iloc[0]
    print(f"  Committee transfer F1: {r['macro_f1']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]")

# Models that improved under transfer on Indian
improved = excl_llama_i[excl_llama_i['delta_f1'] < 0]
print(f"  Models where transfer ≥ local: {', '.join(improved['model_short'].tolist())}")

# ── Rationale portability ──
print("\nRationale portability (transfer rationale vs transfer label-only):")
print("  See rationale portability analysis in cross_institutional_analysis notebook")

# ── Robust models ──
print("\nTransfer-robust models (non-sig in both directions, rationale, long):")
mimic_nonsig = set(mimic_ra[~mimic_ra['sig']]['model'].tolist())
indian_nonsig = set(indian_ra[~indian_ra['sig']]['model'].tolist())
both_nonsig = mimic_nonsig & indian_nonsig
print(f"  {', '.join(short_name(m) for m in both_nonsig)}")


In [ ]:
print('=== Generated outputs ===')
for f in sorted(OUTPUT_DIR.glob('fig_cross*')):
    print(f'  {f.name}')
